### Assignment: Image Classification with Convolutional Neural Networks (CNNs) **[30 points]**

## Objective
In this assignment, you will learn how to build, train, and experiment with **Convolutional Neural Networks (CNNs)** for image classification. You will start with a basic CNN and progressively explore architectural changes and hyperparameter tuning to improve performance.

---

## Part 1: Dataset Selection and Preprocessing

You will use the **CIFAR-10 dataset**, which consists of 60,000 32x32 color images in 10 classes, with 6,000 images per class.

### Task:
- Load the CIFAR-10 dataset using `tensorflow.keras.datasets`
- Normalize the images (scale pixel values to [0, 1])
- Convert the labels to one-hot encoded format (optional but useful for categorical outputs)

```python
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
X_train, X_test = X_train / 255.0, X_test / 255.0
y_train, y_test = to_categorical(y_train), to_categorical(y_test)
```

---

## Part 2: Build a Baseline CNN

### Task:
- Create a CNN model using Keras
- Use Conv2D, MaxPooling2D, Flatten, and Dense layers
- Compile and train the model
- Evaluate accuracy on test data

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)
```

---

## Part 3: Experimentation and Architecture Tuning

### Tasks:
Try modifying the following aspects of your CNN to see how they affect performance:
- Vary the number of convolutional layers (e.g., 2, 3, or more)
- Use different filter sizes (3x3, 5x5)
- Increase the number of filters (e.g., 32, 64, 128)
- Try different activation functions (e.g., `relu`, `leaky_relu`)
- Add Dropout layers to prevent overfitting
- Use Batch Normalization to stabilize training
- Experiment with strides and padding (`valid` vs `same`)

Train your model with each variation and compare validation accuracy.

---

## Part 4: Advanced Exploration

### Tasks:
- Implement data augmentation using `ImageDataGenerator`
- Replace fully connected layers with GlobalAveragePooling
- Try a deeper model using residual connections or separable convolutions
- Compare performance using test set metrics and training time

---

## Part 5: Questions

1. How does the depth of the CNN affect training time and performance?
2. What did you observe when changing the number and size of filters?
3. Which tuning change gave you the highest improvement in accuracy and why do you think it worked?
4. Which regularization methods were most effective?
5. How did data augmentation impact overfitting and generalization?

---

## Hints
- Use `model.evaluate()` and `model.predict()` to test performance
- Visualize a few predictions with their ground truth labels to check how well the model is doing
- Save model checkpoints or best-performing models for future use


Part 1: Dataset Selection and Preprocessing

In [6]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    BatchNormalization, GlobalAveragePooling2D, Activation,
    Add, SeparableConv2D
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

In [7]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("One-hot label shape:", y_train_cat.shape)
print("Pixel range:", X_train.min(), "to", X_train.max())

Training set shape: (50000, 32, 32, 3)
Test set shape: (10000, 32, 32, 3)
One-hot label shape: (50000, 10)
Pixel range: 0.0 to 1.0


Part 2: Build a Baseline CNN

In [8]:
def compile_and_train(model, X_tr, y_tr, X_te, y_te, epochs=10, batch_size=64, use_datagen=False):
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    start = time.time()

    if use_datagen:
        datagen = ImageDataGenerator(
            rotation_range=15,
            width_shift_range=0.1,
            height_shift_range=0.1,
            horizontal_flip=True
        )
        datagen.fit(X_tr)

        history = model.fit(
            datagen.flow(X_tr, y_tr, batch_size=batch_size),
            epochs=epochs,
            validation_split=0.0,
            validation_data=(X_te, y_te),
            verbose=1
        )
    else:
        history = model.fit(
            X_tr, y_tr,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=0.2,
            verbose=1
        )

    training_time = time.time() - start
    test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)

    return history, test_loss, test_acc, training_time


def summarize_run(name, history, test_loss, test_acc, training_time):
    best_val_acc = max(history.history.get("val_accuracy", [np.nan]))
    final_train_acc = history.history.get("accuracy", [np.nan])[-1]

    return {
        "model": name,
        "final_train_accuracy": round(float(final_train_acc), 4),
        "best_validation_accuracy": round(float(best_val_acc), 4) if not np.isnan(best_val_acc) else np.nan,
        "test_accuracy": round(float(test_acc), 4),
        "test_loss": round(float(test_loss), 4),
        "training_time_sec": round(float(training_time), 2)
    }

    
def build_baseline_cnn():
    model = Sequential([
        Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(64, activation="relu"),
        Dense(10, activation="softmax")
    ])
    return model


baseline_model = build_baseline_cnn()
baseline_model.summary()

print("\nTraining baseline CNN...")
baseline_history, baseline_test_loss, baseline_test_acc, baseline_time = compile_and_train(
    baseline_model, X_train, y_train_cat, X_test, y_test_cat, epochs=10, batch_size=64
)

results = []
results.append(
    summarize_run("Baseline CNN", baseline_history, baseline_test_loss, baseline_test_acc, baseline_time)
)

print("Baseline test accuracy:", round(baseline_test_acc, 4))

C:\Users\dylan\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_25 (Conv2D)                   │ (None, 30, 30, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_13 (MaxPooling2D)      │ (None, 15, 15, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_4 (Flatten)                  │ (None, 7200)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_14 (Dense)                     │ (None, 64)                  │         460,864 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_15 (Dense)                     │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 462,410 (1.76 MB)

 Trainable params: 462,410 (1.76 MB)

 Non-trainable params: 0 (0.00 B)


Training baseline CNN...
Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - accuracy: 0.4430 - loss: 1.5674 - val_accuracy: 0.5029 - val_loss: 1.4044
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.5608 - loss: 1.2491 - val_accuracy: 0.5780 - val_loss: 1.2327
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.6047 - loss: 1.1386 - val_accuracy: 0.5773 - val_loss: 1.2202
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 36ms/step - accuracy: 0.6338 - loss: 1.0595 - val_accuracy: 0.6099 - val_loss: 1.1385
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 36s 27ms/step - accuracy: 0.6568 - loss: 0.9948 - val_accuracy: 0.6115 - val_loss: 1.1220
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.6745 - loss: 0.9351 - val_accuracy: 0.6334 - val_loss: 1.0725
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - accuracy: 0.6935 - loss: 0.8857 - val_accuracy: 0.6385 - val_loss: 1.0633
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy

Part 3: Experimentation and Architecture Tuning

In [4]:
def build_deeper_cnn():
    model = Sequential([
        Input(shape=(32, 32, 3)),
        Conv2D(32, (3, 3), padding="same", activation="relu"),
        Conv2D(32, (3, 3), padding="same", activation="relu"),
        MaxPooling2D((2, 2)),

        Conv2D(64, (3, 3), padding="same", activation="relu"),
        Conv2D(64, (3, 3), padding="same", activation="relu"),
        MaxPooling2D((2, 2)),

        Flatten(),
        Dense(128, activation="relu"),
        Dense(10, activation="softmax")
    ])
    return model


def build_bn_dropout_cnn():
    model = Sequential([
        Input(shape=(32, 32, 3)),

        Conv2D(32, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        Conv2D(32, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        Conv2D(64, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        Conv2D(64, (3, 3), padding="same"),
        BatchNormalization(),
        Activation("relu"),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        Flatten(),
        Dense(128),
        BatchNormalization(),
        Activation("relu"),
        Dropout(0.5),
        Dense(10, activation="softmax")
    ])
    return model


def build_leakyrelu_cnn():
    model = Sequential([
        Input(shape=(32, 32, 3)),
        Conv2D(32, (3, 3), padding="same"),
        tf.keras.layers.LeakyReLU(negative_slope=0.1),
        MaxPooling2D((2, 2)),

        Conv2D(64, (3, 3), padding="same"),
        tf.keras.layers.LeakyReLU(negative_slope=0.1),
        MaxPooling2D((2, 2)),

        Flatten(),
        Dense(128),
        tf.keras.layers.LeakyReLU(negative_slope=0.1),
        Dense(10, activation="softmax")
    ])
    return model


tuned_builders = [
    ("Deeper CNN", build_deeper_cnn),
    ("BatchNorm + Dropout CNN", build_bn_dropout_cnn),
    ("LeakyReLU CNN", build_leakyrelu_cnn),
]

for name, builder in tuned_builders:
    print(f"\nTraining {name}...")
    model = builder()
    history, test_loss, test_acc, training_time = compile_and_train(
        model, X_train, y_train_cat, X_test, y_test_cat, epochs=10, batch_size=64
    )
    results.append(summarize_run(name, history, test_loss, test_acc, training_time))


Training Deeper CNN...
Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 66s 100ms/step - accuracy: 0.4687 - loss: 1.4689 - val_accuracy: 0.5956 - val_loss: 1.1507
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 98ms/step - accuracy: 0.6370 - loss: 1.0272 - val_accuracy: 0.6654 - val_loss: 0.9568
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 88s 107ms/step - accuracy: 0.7052 - loss: 0.8398 - val_accuracy: 0.7012 - val_loss: 0.8507
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 73s 92ms/step - accuracy: 0.7494 - loss: 0.7078 - val_accuracy: 0.7188 - val_loss: 0.8237
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 91ms/step - accuracy: 0.7895 - loss: 0.6046 - val_accuracy: 0.7340 - val_loss: 0.7877
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 92ms/step - accuracy: 0.8235 - loss: 0.5067 - val_accuracy: 0.7355 - val_loss: 0.7957
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 94ms/step - accuracy: 0.8570 - loss: 0.4119 - val_accuracy: 0.7247 - val_loss: 0.9089
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 91ms/step - accuracy

Part 4: Advanced Exploration

In [5]:
def build_gap_augmented_cnn():
    model = Sequential([
        Input(shape=(32, 32, 3)),

        Conv2D(32, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        Conv2D(32, (3, 3), padding="same", activation="relu"),
        MaxPooling2D((2, 2)),

        Conv2D(64, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        Conv2D(64, (3, 3), padding="same", activation="relu"),
        MaxPooling2D((2, 2)),

        Conv2D(128, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        GlobalAveragePooling2D(),

        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(10, activation="softmax")
    ])
    return model


def residual_block(x, filters):
    shortcut = x

    x = Conv2D(filters, (3, 3), padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, (3, 3), padding="same")(x)
    x = BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, (1, 1), padding="same")(shortcut)
        shortcut = BatchNormalization()(shortcut)

    x = Add()([x, shortcut])
    x = Activation("relu")(x)
    return x


def build_residual_cnn():
    inputs = Input(shape=(32, 32, 3))

    x = Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = residual_block(x, 32)
    x = MaxPooling2D((2, 2))(x)

    x = residual_block(x, 64)
    x = MaxPooling2D((2, 2))(x)

    x = residual_block(x, 128)
    x = GlobalAveragePooling2D()(x)

    x = Dense(64, activation="relu")(x)
    outputs = Dense(10, activation="softmax")(x)

    model = Model(inputs, outputs)
    return model


def build_separable_cnn():
    model = Sequential([
        Input(shape=(32, 32, 3)),

        SeparableConv2D(32, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        SeparableConv2D(64, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        SeparableConv2D(128, (3, 3), padding="same", activation="relu"),
        BatchNormalization(),
        GlobalAveragePooling2D(),

        Dense(64, activation="relu"),
        Dense(10, activation="softmax")
    ])
    return model


advanced_builders = [
    ("Augmentation + GAP CNN", build_gap_augmented_cnn, True),
    ("Residual CNN", build_residual_cnn, False),
    ("SeparableConv CNN", build_separable_cnn, False),
]

for name, builder, use_datagen in advanced_builders:
    print(f"\nTraining {name}...")
    model = builder()
    history, test_loss, test_acc, training_time = compile_and_train(
        model,
        X_train,
        y_train_cat,
        X_test,
        y_test_cat,
        epochs=10,
        batch_size=64,
        use_datagen=use_datagen
    )
    results.append(summarize_run(name, history, test_loss, test_acc, training_time))

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="test_accuracy", ascending=False).reset_index(drop=True)

print("\nModel comparison:")
print(results_df)


Training Augmentation + GAP CNN...
Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 184s 225ms/step - accuracy: 0.4778 - loss: 1.4387 - val_accuracy: 0.4832 - val_loss: 1.4974
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 136s 173ms/step - accuracy: 0.6224 - loss: 1.0693 - val_accuracy: 0.5820 - val_loss: 1.2211
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 388s 496ms/step - accuracy: 0.6792 - loss: 0.9185 - val_accuracy: 0.5896 - val_loss: 1.3101
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 116s 148ms/step - accuracy: 0.7183 - loss: 0.8167 - val_accuracy: 0.6919 - val_loss: 0.9412
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 119s 152ms/step - accuracy: 0.7430 - loss: 0.7457 - val_accuracy: 0.7396 - val_loss: 0.7499
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 121s 154ms/step - accuracy: 0.7588 - loss: 0.7040 - val_accuracy: 0.6627 - val_loss: 1.0410
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 117s 149ms/step - accuracy: 0.7753 - loss: 0.6627 - val_accuracy: 0.7566 - val_loss: 0.7598
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 

Part 5: Questions

1. Based on the results of each CNN, increasing depth generally led to improved accuracy. The Deeper CNN had a final training accuracy of 0.9310 and hte Baseline CNN had a final training accuracy of only 0.6840. However, increasing the depth also greatly increases the computational cost. The Deeper CNN had a training time of 708.43 seconds while the Baseline CNN only had a training time of 116.10 seconds.

2. Adding more filters improved CNN performance over the baseline but also increased training time and the likelyhood of overfitting. In general, more filters meant better representation power but also more computation and need for regularization.

3. The tuning change that resulted in the greatest improvement in test accuracy from the Baseline CNN was the Augmentation combined with GlobalAveragePooling (GAP). Baseline CNN test accuracy was 0.6155 and Augmentation + GAP CNN test accuracy was 0.7659.

4. The BatchNorm + Dropout CNN had a final training accuracy and best validation accuracy of 0.7849 and 0.7814 respectively. This gap is extremely small, meaning that this model is very effective at controlling overfittiing. The Deeper CNN had a final training accuracy of 0.9310 and a best validation accuracy of 0.7355. This shows that while it performed better, it demonstrated a large amount of overfitting.

5. Data augmentation positively affected generalization. It increased validation accuracy, test accuracy, and reduced overfitting.